# 03 Text Preprocessing

In this notebook we prepare the textual data for machine learning models.

Raw text cannot be directly used by most machine learning algorithms, therefore several preprocessing steps are required.

The preprocessing pipeline includes:

- converting text to lowercase
- removing punctuation and special characters
- removing stopwords
- tokenization
- lemmatization

To better evaluate the effect of preprocessing, two versions of cleaned text will be created:

1. **Basic preprocessing** – minimal cleaning of the text  
2. **Advanced preprocessing** – additional normalization and lemmatization

These processed texts will later be used for feature extraction and machine learning models.

In [3]:
import pandas as pd
import re

from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...


True

In [5]:
DATA_PATH = Path("../processed/reuters_raw_documents.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

(5000, 6)


,file_name,author,split,text,text_length,word_count
0,106247newsML.txt,AaronPressman,train,The Internet may be overflowing with new techn...,2009,319
1,120600newsML.txt,AaronPressman,train,The U.S. Postal Service announced Wednesday a ...,2604,416
2,120683newsML.txt,AaronPressman,train,Elementary school students with access to the ...,491,72
3,136958newsML.txt,AaronPressman,train,An influential Internet organisation has backe...,2907,463
4,137498newsML.txt,AaronPressman,train,An influential Internet organisation has backe...,2299,356


In [6]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [7]:
def basic_clean(text):

    text = text.lower()

    text = re.sub(r"[^a-z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
df["clean_text_basic"] = df["text"].apply(basic_clean)

df[["text", "clean_text_basic"]].head()

,text,clean_text_basic
0,The Internet may be overflowing with new techn...,the internet may be overflowing with new techn...
1,The U.S. Postal Service announced Wednesday a ...,the u s postal service announced wednesday a p...
2,Elementary school students with access to the ...,elementary school students with access to the ...
3,An influential Internet organisation has backe...,an influential internet organisation has backe...
4,An influential Internet organisation has backe...,an influential internet organisation has backe...


In [9]:
def advanced_clean(text):

    text = text.lower()

    text = re.sub(r"[^a-z\s]", " ", text)

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [10]:
df["clean_text_advanced"] = df["text"].apply(advanced_clean)

df[["text", "clean_text_advanced"]].head()

,text,clean_text_advanced
0,The Internet may be overflowing with new techn...,internet may overflowing new technology crime ...
1,The U.S. Postal Service announced Wednesday a ...,u postal service announced wednesday plan boos...
2,Elementary school students with access to the ...,elementary school student access internet lear...
3,An influential Internet organisation has backe...,influential internet organisation backed away ...
4,An influential Internet organisation has backe...,influential internet organisation backed away ...


## Text preprocessing

Raw text often contains punctuation, special characters, numbers, and very common words that do not contribute meaningful information for machine learning models.

To prepare the data for analysis, two levels of preprocessing were applied.

### Basic preprocessing

The basic preprocessing pipeline includes:

- converting text to lowercase
- removing punctuation and non-alphabetic characters
- removing extra whitespace

This version preserves most of the original words but ensures a cleaner and more consistent text representation.

### Advanced preprocessing

The advanced preprocessing pipeline applies additional normalization steps:

- removal of common English stopwords
- lemmatization of words to their base form

Lemmatization helps reduce morphological variations of words, allowing terms such as *running*, *runs*, and *ran* to be mapped to a common base form.

These preprocessing steps reduce noise in the dataset and help machine learning models focus on more informative terms that may capture stylistic differences between authors.

## Comparison of preprocessing approaches

The examples above illustrate the difference between the two preprocessing pipelines.

### Basic preprocessing

Basic preprocessing performs minimal normalization of the text:

- conversion to lowercase
- removal of punctuation and non-alphabetic characters
- normalization of whitespace

This version preserves most of the original text content while ensuring consistent formatting.

### Advanced preprocessing

Advanced preprocessing applies additional transformations:

- removal of common English stopwords
- lemmatization of words to their base form

The resulting text is shorter and contains primarily **informative terms**, which are more useful for machine learning models.

For example, grammatical words such as *the*, *with*, and *be* are removed, while meaningful terms like *internet*, *technology*, and *organisation* remain.

Using multiple preprocessing strategies allows us to later compare how different text representations influence the performance of machine learning models.

In [11]:
df["word_count_advanced"] = df["clean_text_advanced"].str.split().str.len()

df["word_count_advanced"].describe()

count    5000.000000
mean      302.457000
std        81.133985
min        33.000000
25%       253.000000
50%       303.000000
75%       347.000000
max       884.000000
Name: word_count_advanced, dtype: float64

In [12]:
comparison = pd.DataFrame({
    "original_words": df["word_count"],
    "after_preprocessing": df["word_count_advanced"]
})

comparison.describe()

,original_words,after_preprocessing
count,5000.000000,5000.000000
mean,505.660400,302.457000
std,135.473621,81.133985
min,51.000000,33.000000
25%,425.000000,253.000000
50%,510.000000,303.000000
75%,579.000000,347.000000
max,1499.000000,884.000000


## Impact of preprocessing on document length

To evaluate the effect of text preprocessing, we compared the number of words in the original documents with the number of words after applying the advanced preprocessing pipeline.

The results show a significant reduction in document length:

- **Average words before preprocessing:** ~506  
- **Average words after preprocessing:** ~302  

This corresponds to a reduction of approximately **40% of the original tokens**.

The reduction occurs primarily because common stopwords (such as *the*, *and*, *to*, etc.) are removed during preprocessing. These words appear very frequently in natural language but typically provide little useful information for distinguishing documents.

By removing such words and applying lemmatization, the resulting text representation becomes more compact and focuses on **informative terms that better capture the semantic and stylistic characteristics of the documents**.

Reducing noise in the dataset is an important step before applying feature extraction methods such as TF-IDF and training machine learning models.

In [13]:
OUTPUT_DIR = Path("../processed")
OUTPUT_DIR.mkdir(exist_ok=True)

df.to_csv(OUTPUT_DIR / "reuters_preprocessed.csv", index=False)

print("Saved:", OUTPUT_DIR / "reuters_preprocessed.csv")
print("Shape:", df.shape)

Saved: ..\processed\reuters_preprocessed.csv
Shape: (5000, 9)


## Summary of text preprocessing

In this notebook we prepared the raw text data for machine learning analysis.

Two preprocessing pipelines were implemented:

1. **Basic preprocessing** – minimal normalization of the text  
2. **Advanced preprocessing** – removal of stopwords and lemmatization

The advanced preprocessing pipeline significantly reduced the number of words in the documents, removing approximately **40% of the original tokens**.

This step helps reduce noise and allows machine learning algorithms to focus on informative words that better capture the content and writing style of the documents.

The processed dataset was saved to disk so it can be reused in the following stages of the project.

In the next notebook we will perform **feature engineering**, where the cleaned text will be transformed into numerical representations using techniques such as **TF-IDF** before training machine learning models.